In [ ]:
# ⚠️  OLD CELL — superseded. Run cell 2117b490 below instead.
# (Paths here point to deleted data — this cell will raise if run.)
raise RuntimeError("Run the updated training cell (2117b490) below — not this one.")

In [ ]:
# ── Show final/peak validation Dice for the latest v3 run ─────────────────────
from pathlib import Path
import os, csv, json, re

# Point to your v3 root
RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train")

def latest_run_dir(run_root: Path) -> Path:
    runs_dir = run_root / "runs"
    candidates = [p for p in runs_dir.glob("*") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No run folders found under {runs_dir}")
    # pick most recently modified run folder
    return max(candidates, key=lambda p: p.stat().st_mtime)

def read_from_history_csv(cb_dir: Path):
    csv_path = cb_dir / "history.csv"
    if not csv_path.exists():
        return None
    best_val = best_epoch = last_val = last_epoch = None
    with open(csv_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f)):
            v = row.get("val_dice_coefficient")
            if not v:
                continue
            val = float(v)
            last_val, last_epoch = val, i
            if best_val is None or val > best_val:
                best_val, best_epoch = val, i
    if last_val is None:
        return None
    return {
        "source": "CSV",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(csv_path),
    }

def read_from_history_json(cb_dir: Path):
    jpath = cb_dir / "artifacts" / "history_epoch.json"
    if not jpath.exists():
        return None
    with open(jpath) as f:
        h = json.load(f)
    vals = h.get("val_dice_coefficient") or []
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "JSON",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(jpath),
    }

def read_from_log(log_dir: Path):
    log_path = log_dir / "train_stdout_stderr.log"
    if not log_path.exists():
        return None
    pat = re.compile(r"val_dice_coefficient:\s*([0-9]*\.?[0-9]+)")
    vals = []
    with open(log_path, "r", errors="ignore") as f:
        for line in f:
            m = pat.search(line)
            if m:
                vals.append(float(m.group(1)))
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "LOG",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(log_path),
    }

# Resolve the latest run under v3
run_dir = latest_run_dir(RUN_ROOT)
cb_dir  = run_dir / "callbacks"
log_dir = run_dir / "logs"

# Prefer CSV → JSON → logs
res = (read_from_history_csv(cb_dir)
       or read_from_history_json(cb_dir)
       or read_from_log(log_dir))

print(f"Using RUN_DIR: {run_dir}")
if res:
    print(f"[{res['source']}] last val_dice: {res['last_val']:.6f} (epoch {res['last_epoch']})")
    print(f"[{res['source']}] best  val_dice: {res['best_val']:.6f} (epoch {res['best_epoch']})")
    print(f"Source file: {res['path']}")
else:
    print("Could not find val_dice in CSV, JSON, or logs for this run.")


# Train v3.1 resume v3 training

In [ ]:
# === ARC_ATLAS_Train_v3_Low_Quality — train on lowest-quality 522 non-held-out cases ===
from pathlib import Path
import importlib.util, os, sys, gc, time, traceback, shlex, subprocess

# --------- Paths ----------
PREFERRED_GPU = "0"   # set to None to auto-pick the GPU with the most free VRAM
MIN_FREE_MB = 12000   # fail fast if the selected GPU has less than this free

SPLIT_ROOT  = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_LowQualityTrain_Split_Data/train_low_quality").parent
TRAIN_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_LowQualityTrain_Split_Data/train_low_quality")
TRAIN_T1    = TRAIN_DIR / "t1"
TRAIN_MASKS = TRAIN_DIR / "masks"
SPLIT_SUMMARY = SPLIT_ROOT / "split_summary.json"

RUN_ROOT   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train")
MODULE_PATH = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")

def _query_gpus():
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,memory.free,memory.total",
        "--format=csv,noheader,nounits",
    ]
    out = subprocess.check_output(cmd, text=True)
    rows = []
    for line in out.strip().splitlines():
        idx, free_mb, total_mb = [x.strip() for x in line.split(",")]
        rows.append({
            "index": idx,
            "free_mb": int(free_mb),
            "total_mb": int(total_mb),
        })
    return rows

def _query_gpu_processes():
    cmd = [
        "nvidia-smi",
        "--query-compute-apps=pid,gpu_uuid,used_memory,process_name",
        "--format=csv,noheader,nounits",
    ]
    try:
        out = subprocess.check_output(cmd, text=True)
    except Exception:
        return []
    rows = []
    for line in out.strip().splitlines():
        if not line.strip():
            continue
        pid, gpu_uuid, used_mb, process_name = [x.strip() for x in line.split(",", 3)]
        rows.append({
            "pid": pid,
            "gpu_uuid": gpu_uuid,
            "used_mb": used_mb,
            "process_name": process_name,
        })
    return rows

def _select_gpu(preferred_gpu, min_free_mb):
    gpus = _query_gpus()
    if not gpus:
        raise RuntimeError("No GPUs reported by nvidia-smi.")

    selected = None
    if preferred_gpu is not None:
        selected = next((g for g in gpus if g["index"] == str(preferred_gpu)), None)
        if selected is None:
            raise RuntimeError(f"Preferred GPU {preferred_gpu} not found. Visible GPUs: {gpus}")
    else:
        selected = max(gpus, key=lambda g: g["free_mb"])

    if selected["free_mb"] < min_free_mb:
        proc_rows = _query_gpu_processes()
        proc_text = "\n".join(
            f"  pid={r['pid']} used={r['used_mb']}MB name={r['process_name']}"
            for r in proc_rows
        ) or "  <none>"
        gpu_text = "\n".join(
            f"  GPU {g['index']}: free={g['free_mb']}MB / total={g['total_mb']}MB"
            for g in gpus
        )
        raise RuntimeError(
            "Not enough free GPU memory to start training.\n"
            f"Selected GPU {selected['index']} has only {selected['free_mb']}MB free; "
            f"require at least {min_free_mb}MB.\n"
            "Current GPU memory:\n"
            f"{gpu_text}\n"
            "Active GPU compute processes:\n"
            f"{proc_text}\n"
            "Stop or restart the stale TensorFlow/Jupyter kernels, then restart this notebook kernel "
            "and rerun this cell."
        )

    return selected["index"]

if "tensorflow" in sys.modules:
    raise RuntimeError(
        "TensorFlow is already imported in this notebook kernel. Restart the kernel before running "
        "this training cell so CUDA_VISIBLE_DEVICES and memory-growth settings apply cleanly."
    )

CUDA_ID = _select_gpu(PREFERRED_GPU, MIN_FREE_MB)

# --------- Env setup (must happen BEFORE TensorFlow import) ----------
os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_ID)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

# Import TF only after device selection
import tensorflow as tf
from tensorflow.keras import mixed_precision

# --------- New run folders ----------
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
LOG_DIR = RUN_DIR / "logs"
for d in (MODEL_DIR, CALLBACKS_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ["SMARTSOTA_LOG_DIR"] = str(LOG_DIR)

tf.keras.backend.clear_session()
gc.collect()
mixed_precision.set_global_policy("mixed_float16")

class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
            s.flush()
        return len(data)
    def flush(self):
        for s in self.streams:
            s.flush()

log_file = open(LOG_DIR / "train_stdout_stderr.log", "a", buffering=1)
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

print("Run ID:", RUN_ID)
print("Selected GPU:", CUDA_ID)
print("TF:", tf.__version__)
print("GPUs visible to TF:", tf.config.list_physical_devices("GPU"))
print(f"Train images: {len(list(TRAIN_T1.glob('*.nii.gz')))}  masks: {len(list(TRAIN_MASKS.glob('*.nii.gz')))}")
print("Split summary:", SPLIT_SUMMARY)

for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("set_memory_growth failed:", e)

spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
seg.tf = tf
spec.loader.exec_module(seg)

seg.strategy = tf.distribute.get_strategy()
print("Strategy:", type(seg.strategy).__name__)

INPUT_SHAPE   = (192, 224, 192, 1)
BATCH_SIZE    = 1
BASE_FILTERS  = 8
SAM_HEADS     = 2
AUG_INTENSITY = 0.30
VAL_SPLIT     = 0.15
TOTAL_EPOCHS  = 140
INITIAL_EPOCH = 0

INITIAL_LR    = 1e-4
MIN_LR        = 5e-7
WARMUP_EPOCHS = 15

try:
    history = seg.train_dynamic_model(
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,
        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,
        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,
        INPUT_SHAPE=INPUT_SHAPE,
        BATCH_SIZE=BATCH_SIZE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        RESAMPLE_TO_TARGET=True,
        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        VALIDATION_SPLIT=VAL_SPLIT,
        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,
    )
    print("Training complete. Logged keys:", list(getattr(history, "history", {}).keys()))
    print("Run artifacts at:", RUN_DIR)
except Exception:
    print("\n================= UNCAUGHT EXCEPTION =================")
    traceback.print_exc()
    print("======================================================\n")
    try:
        print("Last few GPU snapshots:")
        for _ in range(3):
            subprocess.run(shlex.split("nvidia-smi"), check=False)
            time.sleep(1)
    except Exception:
        pass
    raise
finally:
    try:
        log_file.flush()
    except Exception:
        pass


2026-04-16 10:58:05.045868: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Run ID: 20260416_105806
Selected GPU: 0
TF: 2.20.0
GPUs visible to TF: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Train images: 522  masks: 522
Split summary: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_LowQualityTrain_Split_Data/split_summary.json
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy: _DefaultDistributionStrategy
Strategy: _DefaultDistributionStrategy


2026-04-16 10:58:07,176 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2026-04-16 10:58:07,176 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2026-04-16 10:58:07,176 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.6
- GPU devices: 1
2026-04-16 10:58:07,179 - SmartSOTA_Dynamic - INFO - 🧭 INPUT_SHAPE set to: (192, 224, 192, 1)
2026-04-16 10:58:07,179 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
I0000 00:00:1776358687.283258 3589270 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1776358687.284404 3589270 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
2026-04-16 10:58:07,287 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.78GB | GPU mem track

2026-04-16 11:00:11,846 - SmartSOTA_Dynamic - INFO - Model: "SmartSOTA_Dynamic"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 192, 224,  │          0 │ -                 │
│ (InputLayer)        │ 192, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_block │ (None, 192, 224,  │      2,024 │ input_layer[0][0] │
│ (ResidualConvBlock) │ 192, 8)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block  │ (None, 192, 224,  │      7,192 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 192, 8)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

Epoch 1/140


...

Epoch 140: dice=0.4041 val_dice=0.4291 loss=0.3626 val_loss=0.3476 lr=5.00e-07
417/417 ━━━━━━━━━━━━━━━━━━━━ 214s 513ms/step - dice_coefficient: 0.4041 - loss: 0.3626 - val_dice_coefficient: 0.4291 - val_loss: 0.3476 - learning_rate: 5.0000e-07


2026-04-16 19:24:05,576 - SmartSOTA_Dynamic - INFO - 💾 Saved final weights to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train/runs/20260416_105806/models/smart_sota_dynamic_20260416_105807.final.weights.h5


2026-04-16 19:24:05,971 - SmartSOTA_Dynamic - INFO - 💾 Saved full model to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train/runs/20260416_105806/models/smart_sota_dynamic_20260416_105807.keras
2026-04-16 19:24:05,971 - SmartSOTA_Dynamic - INFO - 🏁 Training complete.


Training complete. Logged keys: ['dice_coefficient', 'loss', 'val_dice_coefficient', 'val_loss', 'learning_rate']
Run artifacts at: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train/runs/20260416_105806
5dc3a68c-e34e-4080-9c3e-2a532b2ccb4d6.30.15dc3a68c-e34e-4080-9c3e-2a532b2ccb4d


: 

In [ ]:
# ── Show final/peak validation Dice for the latest v3 run ─────────────────────
from pathlib import Path
import os, csv, json, re

# Point to your v3 root
RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train")

def latest_run_dir(run_root: Path) -> Path:
    runs_dir = run_root / "runs"
    candidates = [p for p in runs_dir.glob("*") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No run folders found under {runs_dir}")
    # pick most recently modified run folder
    return max(candidates, key=lambda p: p.stat().st_mtime)

def read_from_history_csv(cb_dir: Path):
    csv_path = cb_dir / "history.csv"
    if not csv_path.exists():
        return None
    best_val = best_epoch = last_val = last_epoch = None
    with open(csv_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f)):
            v = row.get("val_dice_coefficient")
            if not v:
                continue
            val = float(v)
            last_val, last_epoch = val, i
            if best_val is None or val > best_val:
                best_val, best_epoch = val, i
    if last_val is None:
        return None
    return {
        "source": "CSV",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(csv_path),
    }

def read_from_history_json(cb_dir: Path):
    jpath = cb_dir / "artifacts" / "history_epoch.json"
    if not jpath.exists():
        return None
    with open(jpath) as f:
        h = json.load(f)
    vals = h.get("val_dice_coefficient") or []
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "JSON",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(jpath),
    }

def read_from_log(log_dir: Path):
    log_path = log_dir / "train_stdout_stderr.log"
    if not log_path.exists():
        return None
    pat = re.compile(r"val_dice_coefficient:\s*([0-9]*\.?[0-9]+)")
    vals = []
    with open(log_path, "r", errors="ignore") as f:
        for line in f:
            m = pat.search(line)
            if m:
                vals.append(float(m.group(1)))
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "LOG",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(log_path),
    }

# Resolve the latest run under v3
run_dir = latest_run_dir(RUN_ROOT)
cb_dir  = run_dir / "callbacks"
log_dir = run_dir / "logs"

# Prefer CSV → JSON → logs
res = (read_from_history_csv(cb_dir)
       or read_from_history_json(cb_dir)
       or read_from_log(log_dir))

print(f"Using RUN_DIR: {run_dir}")
if res:
    print(f"[{res['source']}] last val_dice: {res['last_val']:.6f} (epoch {res['last_epoch']})")
    print(f"[{res['source']}] best  val_dice: {res['best_val']:.6f} (epoch {res['best_epoch']})")
    print(f"Source file: {res['path']}")
else:
    print("Could not find val_dice in CSV, JSON, or logs for this run.")
